In [1]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
from ingest import ingest_faq_data,build_index
from rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

load_dotenv()

system_instruction = """
Your task is to answer the questions from the course participants based on the given context
    Use the context to find relevant information and provide accurate answer to the user
    if there is no answer in the context do not make up things just tell the user that you don't know
"""

docs = ingest_faq_data()
index = TextSearchIndex(text_fields=["question", "section", "answer"],
    keyword_fields=["course"],db_path="faq.db")

gemini_client = genai.Client()

config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instruction
)

assistant = RAGBase(index = index,llm_client = gemini_client,config = config ,model = "gemini-3.6-flash")
question = "I just discovered the course. Can i join it?"
answer = assistant.rag(question)
print(answer)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [19]:
def search(question,num_results = 5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
            question,
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=num_results
    )


search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters_json_schema={
        "type":"object",
        "properties" : {
            "question": {
                "type":"string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required" : ["question"],
        "additionalProperties": False
    }
)

search_tool = types.Tool(function_declarations=[search_declaration])

In [21]:
tool_config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instruction,
    tools = [search_tool]
)

contents = [
    types.Content(role = "user", parts = [types.Part(text=question)])
]

response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)

response.function_calls

[FunctionCall(
   args={
     'question': 'Can I join the course after it started?'
   },
   id='call_38333',
   name='search'
 )]

In [26]:
call = response.function_calls[0]
args = call.args
print(args)
name = call.name
print(name)
print(response.candidates[0].content)

result = search(**args)
print(result)


contents.append(response.candidates[0].content)

contents.append(
    types.Content(
    role = "user",
    parts= [types.Part.from_function_response(
        name=call.name,
        response={"result": result},
    )
    ],
)
)

{'question': 'Can I join the course after it started?'}
search
parts=[Part(
  function_call=FunctionCall(
    args={
      'question': 'Can I join the course after it started?'
    },
    id='call_38333',
    name='search'
  ),
  thought_signature=b"\x12\xe2\x01\n\xdf\x01\x01i\x14}\x13\xd6\x96\xdf\xae\xdf\x95\x07\x83-z\xc9Q-\xb3n\xf3\x0b\xed\xd2\xdd\x8f\x15\xec\xcc\xc0%\xe7\xee<\x98N\xb1\x91\xba\xf6\x80\x87\xf8\xb3&rI\xce\xf3\xd0S\xbb\xcd\xa8'\xac\xc2Pq\x93a\xfd\xb2?\xd9\xd6\xbb\xa1u\x80E\x92\x82r>\xe8B\x9d8`Q\xd3\x1f\x90]\xef\xe7\xc7\xd3\x1e\xc8...'
)] role='model'
[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course

In [27]:
response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)
print(response.text)

Yes, you can still join! However, if you want to receive a certificate, you will need to submit your project while project submissions are still being accepted.
